#### Initialize

In [1]:
import sys
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
PARENT = HERE.parent.parent.parent  # server/scripts
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))
LEVEL2_BUCKET_PATH = PARENT / "server/out/places_level2"
LEVEL3_BUCKET_PATH = PARENT / "server/out/places_level3"
LEVEL3_BUCKET_PATH.mkdir(parents=True, exist_ok=True)
LEVEL2_BUCKET = [f for f in LEVEL2_BUCKET_PATH.rglob("*.csv") if f.is_file()]
DF_LEVEL2 = pd.concat([pd.read_csv(f) for f in LEVEL2_BUCKET], ignore_index=True)

#### Check Validity

In [2]:
df_validity = pd.DataFrame({ 
    col: DF_LEVEL2[col].notna().sum() / len(DF_LEVEL2) 
    for col in DF_LEVEL2.columns 
}, index=[0])
display(df_validity)

,id,displayName,primaryTypeDisplayName,rating,userRatingCount,location,shortFormattedAddress,googleMapsUri,priceRange,priceLevel,...,accessibilityOptions,addressDescriptor,postalAddress,tile_id,tile_path_id,seed_index,level,predictedType,cuisineType,venueType
0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.851818,0.367094,...,0.695845,0.999618,0.996868,1.0,1.0,1.0,1.0,0.664452,1.0,1.0


#### Parse JSON

In [3]:
import ast
df_level3 = DF_LEVEL2.copy()
valid_postaladdress = df_level3.loc[df_level3["postalAddress"].notna(), "postalAddress"]
valid_postaladdress = valid_postaladdress.apply(ast.literal_eval)
valid_addresscomponents = df_level3.loc[df_level3["addressComponents"].notna(), "addressComponents"]
valid_addresscomponents = valid_addresscomponents.apply(ast.literal_eval)
valid_addressdescriptor = df_level3.loc[df_level3["addressDescriptor"].notna(), "addressDescriptor"]
valid_addressdescriptor = valid_addressdescriptor.apply(ast.literal_eval)
valid_regularopeninghours = df_level3.loc[df_level3["regularOpeningHours"].notna(), "regularOpeningHours"]
valid_regularopeninghours = valid_regularopeninghours.apply(ast.literal_eval)
valid_containingplaces = df_level3.loc[df_level3["containingPlaces"].notna(), "containingPlaces"]
valid_containingplaces = valid_containingplaces.apply(ast.literal_eval)
valid_pricerange = df_level3.loc[df_level3["priceRange"].notna(), "priceRange"]
valid_pricerange = valid_pricerange.apply(ast.literal_eval)

df_level3["location"] = df_level3["location"].apply(ast.literal_eval)
df_level3["postalAddress"] = valid_postaladdress
df_level3["addressComponents"] = valid_addresscomponents
df_level3["addressDescriptor"] = valid_addressdescriptor
df_level3["regularOpeningHours"] = valid_regularopeninghours
df_level3["containingPlaces"] = valid_containingplaces
df_level3["priceRange"] = valid_pricerange

In [4]:
ROW1 = df_level3.iloc[0]
# display(ROW1["postalAddress"])
df_level3["pcd"] = df_level3["postalAddress"].apply(lambda x: x.get("postalCode") if isinstance(x, dict) else "")
df_level3["areacode"] = df_level3["pcd"].apply(lambda x: x.split(" ")[0] if isinstance(x, str) and " " in x else "")
df_level3["lat"] = df_level3["location"].apply(lambda x: x.get("latitude") if isinstance(x, dict) else None)
df_level3["lon"] = df_level3["location"].apply(lambda x: x.get("longitude") if isinstance(x, dict) else None)
df_level3["startPrice"] = df_level3['priceRange'].apply(lambda x: x.get("startPrice", {}).get("units") if isinstance(x, dict) else None).astype(float)
df_level3["endPrice"] = df_level3['priceRange'].apply(lambda x: x.get("endPrice", {}).get("units") if isinstance(x, dict) else None).astype(float)
df_level3["medianPrice"] = df_level3.apply(lambda row: (row["startPrice"] + row["endPrice"]) / 2 if pd.notna(row["startPrice"]) and pd.notna(row["endPrice"]) else None, axis=1)

In [7]:
df_level3_2 = df_level3[df_level3["businessStatus"]=="OPERATIONAL"].copy().reset_index(drop=True)
SCOPE = ["id", "displayName", "primaryTypeDisplayName", "pcd", "lat", "lon", 
    "rating", "userRatingCount", "shortFormattedAddress", "areacode", "googleMapsUri", 
    "websiteUri", "types", 'primaryType', 'tile_id', 'seed_index', 'tile_path_id', 
    'cuisineType', 'venueType', 'priceLevel', 'startPrice', 'endPrice', 'medianPrice']
df_level3_2 = df_level3_2[SCOPE].copy()

#### Export

In [8]:
for seed_id, group in df_level3_2.groupby("seed_index"):
    save_path = LEVEL3_BUCKET_PATH / f"{seed_id}.csv"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    group.to_csv(save_path, index=False)